# Lab 5 Student Version (Google Colab)
## Scheduled Training with Learning Rate Scheduler (StepLR)

This notebook is a **student-learning version** of the attached Lab 5 guide. It preserves **every topic and step** (Tasks 1–3 + Lab Review) and adds explanations of **what you are learning** and **why each step matters**.

## Lab overview
In this lab, you will explore how **learning rate schedulers** can influence model training performance. You will train a simple neural network on the **Fashion-MNIST** dataset:
1) **with** a StepLR learning rate scheduler, and  
2) **without** scheduling,  
then compare training loss and accuracy curves. You will also **visualize learning rate changes over epochs**.

### In this lab, you will
- Apply a **StepLR** learning rate scheduler during training
- Visualize learning rate changes over epochs
- Compare training loss and accuracy **with vs. without** learning-rate scheduling

### Estimated completion time
- **30 minutes**

### Runtime type
- Set runtime type to: **CPU**


# Task 0 (Colab setup): Imports and environment check

### What you are learning
Before training, you verify your environment and imports. This reduces setup issues and makes your results easier to reproduce.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Runtime recommendation for this lab: CPU")


# Task 1: Train a model with StepLR and track learning rate

### What you are learning
You are learning how to attach a **learning rate scheduler** to an optimizer. A scheduler changes the learning rate over time using a rule.  
Here, **StepLR** reduces the learning rate by a factor (`gamma`) every fixed number of epochs (`step_size`). Lowering the learning rate later in training can help the model **converge more smoothly**.

### Goal
Train a simple MLP on Fashion-MNIST while:
- applying StepLR scheduling
- tracking the learning rate used each epoch
- printing the learning rate over epochs

### Steps (from the lab)
1. Import necessary libraries.
2. Load Fashion-MNIST.
3. Define a simple MLP.
4. Initialize model, optimizer, scheduler, criterion.
5. Train and track learning rate per epoch.
6. Review the output.


In [ ]:
# 2. Load Fashion-MNIST.
transform = transforms.ToTensor()

trainset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True
)

print("Training samples:", len(trainset))


In [ ]:
# 3. Define a simple MLP.
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

# Quick sanity check (single forward pass)
model = MLP()
x_demo, _ = next(iter(trainloader))
y_demo = model(x_demo[:2])
print("Demo batch input shape:", x_demo[:2].shape)
print("Demo output shape:", y_demo.shape)


In [ ]:
# 4. Initialize model, optimizer, scheduler, criterion.
model = MLP()
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = StepLR(optimizer, step_size=5, gamma=0.1)
criterion = nn.CrossEntropyLoss()

print("Initial learning rate:", optimizer.param_groups[0]['lr'])
print("StepLR settings -> step_size=5, gamma=0.1")


In [ ]:
# 5. Train model and track LR.
# NOTE: We record the LR used for each epoch BEFORE scheduler.step(),
# so the plot shows the LR that was used during training of that epoch.
lr_per_epoch = []

num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in trainloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Track LR for this epoch (the LR used while training this epoch)
    lr_per_epoch.append(optimizer.param_groups[0]['lr'])

    # Step the scheduler at the end of the epoch (updates LR for the NEXT epoch)
    scheduler.step()

    print(f"Epoch {epoch+1}: Learning Rate = {optimizer.param_groups[0]['lr']:.5f} | Avg Loss = {running_loss/len(trainloader):.4f}")


### Task 1 explanation (learning takeaways)
- `optimizer = SGD(..., lr=0.1)` sets the **starting** learning rate.
- `StepLR(optimizer, step_size=5, gamma=0.1)` means:
  - every **5 epochs**, multiply learning rate by **0.1**
- `scheduler.step()` updates the learning rate according to the schedule.
- Tracking `lr_per_epoch` lets you visualize the schedule in the next task.


# Task 2: Visualize learning rate changes over epochs

### What you are learning
You are learning how to plot the learning rate schedule so you can **see** when and how the scheduler changes the learning rate.  
Visualizing this helps you connect:
- scheduler settings (`step_size`, `gamma`)
- to the actual learning rate values used during training.

### Steps (from the lab)
1. Plot to visualize the learning rate schedule.
2. Review the output plot.


In [ ]:
# 1. Plot to visualize the learning rate schedule.
plt.figure(figsize=(6, 4))
plt.plot(lr_per_epoch, marker='o')
plt.title("Learning Rate per Epoch using StepLR")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.grid(True)
plt.show()


### Task 2 explanation (learning takeaways)
- The plot should show a **step-like** pattern.
- With `step_size=5` and `gamma=0.1`, the LR drops by 10× every 5 epochs.
- This schedule often helps training transition from faster learning (early) to fine-tuning (late).


# Task 3: Compare training loss and accuracy with vs. without scheduling

### What you are learning
You are learning how to run a controlled experiment:
- Train the **same model** twice,
- keep everything else the same,
- change only one factor: **use_scheduler = True/False**.

Then you compare:
- training **accuracy** curves
- training **loss** curves

This is a practical way to understand whether scheduling improves convergence or stability.

### Steps (from the lab)
1. Define reusable training function.
2. Train with scheduler.
3. Train without scheduler.
4. Plot accuracy comparison.
5. Plot loss comparison.
6. Review the outputs.


In [ ]:
# 1. Define reusable training function.
def train_model(use_scheduler: bool = False, num_epochs: int = 15):
    model = MLP()
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    scheduler = StepLR(optimizer, step_size=5, gamma=0.1) if use_scheduler else None
    criterion = nn.CrossEntropyLoss()

    train_loss = []
    train_acc = []

    for epoch in range(num_epochs):
        model.train()
        correct, total, running_loss = 0, 0, 0.0

        for inputs, labels in trainloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        # Step scheduler once per epoch (if enabled)
        if use_scheduler:
            scheduler.step()

        train_loss.append(running_loss / len(trainloader))
        train_acc.append(correct / total)

    return train_loss, train_acc


In [ ]:
# 2. Train with scheduler.
print("Training with StepLR scheduler...")
loss_with, acc_with = train_model(use_scheduler=True, num_epochs=15)

# 3. Train without scheduler.
print("Training WITHOUT scheduler...")
loss_without, acc_without = train_model(use_scheduler=False, num_epochs=15)


In [ ]:
# 4. Plot accuracy comparison.
plt.figure(figsize=(6, 4))
plt.plot(acc_with, label="With StepLR", marker='o')
plt.plot(acc_without, label="Without Scheduler", marker='x')
plt.title("Training Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# 5. Plot loss comparison.
plt.figure(figsize=(6, 4))
plt.plot(loss_with, label="With StepLR", marker='o')
plt.plot(loss_without, label="Without Scheduler", marker='x')
plt.title("Training Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()


### Task 3 explanation (learning takeaways)
- This comparison isolates the effect of scheduling.
- You may observe that scheduling can:
  - stabilize training later (smaller learning rate → smaller parameter updates)
  - improve convergence behavior (loss decreases more smoothly)
- If results look similar, that’s still informative: it means *for this setup*, scheduling might not be a big factor, or you may need different scheduler settings.


# Lab review

1. What is the purpose of the StepLR scheduler?  
A. It increases the learning rate every few steps  
B. It adjusts batch size dynamically  
C. It reduces the learning rate by a factor at regular intervals  
D. It stops training early  

2. Learning rate schedulers help improve training performance by adjusting the learning rate dynamically.  
A. True  
B. False  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answers (click to expand)</strong></summary>

1. **C**  
2. **A (True)**

</details>
